In [ ]:
import pandas as pd
import re, unidecode, sqlalchemy as sa
from sqlalchemy.orm import Session
from sqlalchemy.dialects.postgresql import insert
from tqdm.auto import tqdm
from typing import Dict, List, Tuple, Any

from apps.ingestion.seed_and_ingest import Player, FootballNews, Base, get_engine

DB_URL = "postgresql+psycopg2://scout:scout@db:5432/scouting"
engine = sa.create_engine(DB_URL)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df_players = pd.read_sql_table("players", con=engine)

In [ ]:
df_players.head()

In [ ]:
df_players = df_players.fillna(0)

In [ ]:
df_players["gk_goals_against_per90"] = df_players["gk_goals_against"] / df_players["minutes_90s"]

In [ ]:
df_players.fillna({"gk_goals_against_per90":0.0}, inplace=True)

In [ ]:
stats_cols = ['goals', 'assists',
       'expected_goals', 'expected_assists',
       'no_penalty_expected_goals_plus_expected_assists',
       'progressive_carries', 'progressive_passes',
       'progressive_passes_received', 'goals_per90', 'assists_per90',
       'goals_assists_per90', 'expected_goals_per90', 'expected_assists_per90',
       'expected_goals_assists_per90', 'gk_goals_against_per90', 'gk_pens_allowed',
       'gk_free_kick_goals_against', 'gk_corner_kick_goals_against',
       'gk_own_goals_against', 'gk_psxg',
       'gk_psnpxg_per_shot_on_target_against', 'passes_completed', 'passes',
       'passes_pct', 'passes_progressive_distance', 'passes_completed_long',
       'passes_long', 'passes_pct_long', 'tackles', 'tackles_won',
       'challenge_tackles', 'challenges', 'challenge_tackles_pct',
       'challenges_lost', 'blocks', 'blocked_shots', 'blocked_passes',
       'interceptions', 'tackles_interceptions', 'clearances', 'errors',
       ]

In [ ]:
stats_groups = {
    "GK":['gk_goals_against_per90', 'gk_pens_allowed',
       'gk_free_kick_goals_against', 'gk_corner_kick_goals_against',
       'gk_own_goals_against', 'gk_psxg',
       'gk_psnpxg_per_shot_on_target_against'],
    "DF":['tackles', 'tackles_won',
       'challenge_tackles','challenge_tackles_pct','blocks', 'blocked_shots', 'blocked_passes',
       'interceptions', 'tackles_interceptions','errors'],
    "PASS":['assists','assists_per90', 'expected_assists_per90','passes_pct', 'passes_progressive_distance', 'passes_completed_long',
       'passes_long', 'passes_pct_long'],
    "PACE":['progressive_carries', 'progressive_passes',
       'progressive_passes_received','challenges','challenges_lost','clearances'],
    "ATK": ['goals','expected_goals','no_penalty_expected_goals_plus_expected_assists','goals_assists_per90', 'expected_goals_per90', 'expected_assists_per90',
       'expected_goals_assists_per90']
}

In [ ]:
assert df_players["id"].nunique() == df_players.shape[0]

In [ ]:
index_col = "id"

In [ ]:
stats_df = df_players[stats_cols].fillna(0)

In [ ]:
stats_ranked_df = round(stats_df.rank(pct=True)*100,0)

In [ ]:
for s in stats_cols:
    print(s)
    try:
        stats_df[s].plot(kind="density")
        plt.show();
    except Exception as e:
        print(f"Impossible to show {s} because {e}")

In [ ]:
df_players.assists.unique()

In [ ]:
stats_df.head()

In [ ]:
stats_ranked_df

In [ ]:
df_players[df_players.position == "GK"][stats_groups["GK"]]

In [ ]:
stats_ranked_df[stats_groups["GK"]].head(30)